In [ ]:
import pandas as pd

meta = pd.read_parquet("data/metadata.parquet")
txt = pd.read_parquet("data/BigEarthNet.txt.parquet")

print("Metadata:", meta.shape)
print("Text:", txt.shape)

Metadata: (480038, 8)
Text: (9553962, 13)


In [4]:
print(txt["type"].value_counts())
print(txt["split"].value_counts())

type
binary          3625160
mcq             3259184
bounding box    2205686
captioning       463932
Name: count, dtype: int64
split
train         4674281
validation    2454690
test          2409962
bench           15029
Name: count, dtype: int64


In [5]:
caption = txt[txt["type"] == "captioning"].copy()

print("Linhas de caption:", len(caption))
print("Patches únicos:", caption["patch_id"].nunique())

print("\nPatches do captioning que estão no metadata:",
      caption["patch_id"].isin(meta["patch_id"]).sum())

print("\nPatches únicos do captioning que estão no metadata:",
      caption.loc[caption["patch_id"].isin(meta["patch_id"]), "patch_id"].nunique())

Linhas de caption: 463932
Patches únicos: 463932

Patches do captioning que estão no metadata: 463932

Patches únicos do captioning que estão no metadata: 463932


In [6]:
import pandas as pd

# Carregar
meta = pd.read_parquet("data/metadata.parquet")
txt = pd.read_parquet("data/BigEarthNet.txt.parquet")

# Somente captions
caption = txt[txt["type"] == "captioning"].copy()

# Manter apenas as colunas que nos interessam
caption = caption[
    [
        "patch_id",
        "input",
        "output",
        "split",
        "latitude",
        "longitude",
        "country",
        "season",
        "climate_zone"
    ]
].copy()

# Quantidade desejada por split
n_train = 20000
n_val = 5000
n_test = 5000

# Amostragem
train = caption[caption["split"] == "train"].sample(
    n=n_train,
    random_state=42
)

val = caption[caption["split"] == "validation"].sample(
    n=n_val,
    random_state=42
)

test = caption[caption["split"] == "test"].sample(
    n=n_test,
    random_state=42
)

# Juntar
selected = pd.concat([train, val, test], ignore_index=True)

# Embaralhar
selected = selected.sample(frac=1, random_state=42).reset_index(drop=True)

print(selected.shape)
print(selected["split"].value_counts())
print(selected.head())

(30000, 9)
split
train         20000
validation     5000
test           5000
Name: count, dtype: int64
                                            patch_id  \
0  S2A_MSIL2A_20180225T114351_N9999_R123_T29UPU_0...   
1  S2B_MSIL2A_20171015T104009_N9999_R008_T31UGR_1...   
2  S2A_MSIL2A_20171002T094031_N9999_R036_T34TCR_7...   
3  S2A_MSIL2A_20171101T094131_N9999_R036_T35VNK_6...   
4  S2B_MSIL2A_20180515T112109_N9999_R037_T29SNC_0...   

                                               input  \
0  Explain the content of the image, highlighting...   
1  Explain what can be seen in this satellite ima...   
2  Explain the observed land cover and spatial pa...   
3  Give a comprehensive overview of the image, sp...   
4  Explain the content of the image, highlighting...   

                                              output       split   latitude  \
0  This satellite image, captured during the wint...       train  52.510569   
1  This satellite image, captured during the fall...  validation 

In [7]:
selected.to_csv("data/selected_30k.csv", index=False)